In [2]:
#Menyiapkan Dataset
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:
#Membuat SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window
import pandas as pd

spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca transaksi dari HDFS + tambah kolom pendapatan
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Membuat df_target dari dictionary data_target_cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.show(5)
df_target.show()

26/09/24 14:11:55 WARN Utils: Your hostname, nada resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp1s0)
26/09/24 14:11:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 14:11:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [4]:
#A. Join & Perbandingan Target

#Meringkas total pendapatan per kota dari df_transkasi
ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

#Join dengan df_target
hasil_a = ringkasan.join(df_target, on="kota", how="inner")
hasil_a = hasil_a.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

#Menambahkan kolom pencapaian_persen dan mengurutkan hasilnya menggunakan orderBy
hasil_a = hasil_a.orderBy(col("pencapaian_persen").desc())
hasil_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [5]:
#B. Window FUnction - Kategori Terlaris per Kota

#Kategori dengan pendapatan tertinggi tiap kota (top 1)
ringkasan_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

#Menggunakan row_number
hasil_b = ringkasan_kategori_kota.withColumn(
    "peringkat", row_number().over(window_kota)
).filter(col("peringkat") == 1)

hasil_b.show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



In [6]:
#C. Spark SQL

#Daftarkan df_transaksi dan df_target sebagai temprorary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

#Query SQL yang menampilkan kota, pic_cabang, dan jumlah transaksi (COUNT) di kota tersebut
hasil_c = spark.sql("""
    SELECT t.kota, tg.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [10]:
#D. Kesimpulan
"Berdasarkan hasil A dan B, cabang mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? Sertakan angka-angka pendukung dari hasil analisis!"

"Jawaban: Berdasarkan hasil analisis bagian A dan B, cabang kota Purworejo menunjukkan kinerja paling baik dengan pencapaian sebesar 152.16666666666669% dari target bulanan sebesar 30000000, melampaui target yang ditetapkan. Kategori produk Kesehatan dan Kecantikan menjadi kontributor utama dari cabang tersebut berdasarkan hasil bagian B. Sebaliknya, cabang Semarang perlu mendapatkan perhatian khusus dari manajemen karena hanya mencapai 69.4090909090909% dari target bulanan sebesar 55000000, jauh di bawah cabang lainnya. Manajemen disarankan untuk mengevaluasi dan memperhatikan lebih cabang tersebut agar dapat meningkatkan pencapaian target di periode berikutnya."

'Jawaban: Berdasarkan hasil analisis bagian A dan B, cabang kota Purworejo menunjukkan kinerja paling baik dengan pencapaian sebesar 152.16666666666669% dari target bulanan sebesar 30000000, melampaui target yang ditetapkan. Kategori produk Kesehatan dan Kecantikan menjadi kontributor utama dari cabang tersebut berdasarkan hasil bagian B. Sebaliknya, cabang Semarang perlu mendapatkan perhatian khusus dari manajemen karena hanya mencapai 69.4090909090909% dari target bulanan sebesar 55000000, jauh di bawah cabang lainnya. Manajemen disarankan untuk mengevaluasi dan memperhatikan lebih cabang tersebut agar dapat meningkatkan pencapaian target di periode berikutnya.'

In [11]:
#Menutup SparkSession

spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
